In [1]:
import numpy as np
import json

X1_train = np.load("../models/X1_train.npy")
X1_test  = np.load("../models/X1_test.npy")
y1_train = np.load("../models/y1_train.npy")
y1_test  = np.load("../models/y1_test.npy")

print("Train shape:", X1_train.shape, y1_train.shape)
print("Test shape:", X1_test.shape, y1_test.shape)

Train shape: (59793, 48, 10) (59793, 6, 4)
Test shape: (14949, 48, 10) (14949, 6, 4)


In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_model(y_true, y_pred, name="Model"):
    y_true_flat = y_true.reshape(-1, y_true.shape[-1])
    y_pred_flat = y_pred.reshape(-1, y_pred.shape[-1])

    mse = mean_squared_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    r2 = r2_score(y_true_flat, y_pred_flat)

    print(f"\n{name}")
    print("MSE :", mse)
    print("RMSE:", rmse)
    print("MAE :", mae)
    print("R2  :", r2)

    return mse, rmse, mae, r2

In [3]:
import numpy as np

# Last input timestep (scaled)
last_input_train = X1_train[:, -1, :]
last_input_test  = X1_test[:, -1, :]

# Target indices (adjust if needed)
target_indices = [0,1,2,3]

last_targets_train = last_input_train[:, target_indices]
last_targets_test  = last_input_test[:, target_indices]

# True next step (first horizon)
y_next_train = y1_train[:, 0, :]
y_next_test  = y1_test[:, 0, :]

# Compute delta
delta_train = y_next_train - last_targets_train
delta_test  = y_next_test - last_targets_test

print("Delta shape:", delta_train.shape)

Delta shape: (59793, 4)


In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

delta_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(48, X1_train.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(4)   # Predict delta for 4 targets
])

delta_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_delta = delta_model.fit(
    X1_train,
    delta_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/50


d:\Aditya\Projects\SDP\AquaCulture\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


748/748 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 0.0207 - mae: 0.1106 - val_loss: 0.0210 - val_mae: 0.1159
Epoch 2/50
748/748 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - loss: 0.0117 - mae: 0.0827 - val_loss: 0.0094 - val_mae: 0.0777
Epoch 3/50
748/748 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - loss: 0.0078 - mae: 0.0677 - val_loss: 0.0074 - val_mae: 0.0691
Epoch 4/50
748/748 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - loss: 0.0069 - mae: 0.0632 - val_loss: 0.0069 - val_mae: 0.0666
Epoch 5/50
748/748 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - loss: 0.0063 - mae: 0.0606 - val_loss: 0.0066 - val_mae: 0.0653
Epoch 6/50
748/748 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - loss: 0.0060 - mae: 0.0591 - val_loss: 0.0061 - val_mae: 0.0628
Epoch 7/50
748/748 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - loss: 0.0059 - mae: 0.0582 - val_loss: 0.0061 - val_mae: 0.0628
Epoch 8/50
748/748 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - loss: 0.0057 - mae: 0.0576 - val_loss: 0.0060 - val_mae: 0.0626
Epoch 9/50
748/748 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/st

In [5]:
def iterative_forecast(model, X_input, horizon=6):
    X_current = X_input.copy()
    predictions = []

    for _ in range(horizon):
        delta_pred = model.predict(X_current, verbose=0)

        last_step = X_current[:, -1, :]
        last_targets = last_step[:, target_indices]

        next_pred = last_targets + delta_pred

        predictions.append(next_pred)

        # update input window
        next_full = last_step.copy()
        next_full[:, target_indices] = next_pred

        X_current = np.concatenate(
            [X_current[:, 1:, :], next_full[:, np.newaxis, :]],
            axis=1
        )

    return np.stack(predictions, axis=1)

y_pred_delta = iterative_forecast(delta_model, X1_test, horizon=6)

print("Delta prediction shape:", y_pred_delta.shape)

Delta prediction shape: (14949, 6, 4)


In [6]:
evaluate_model(y1_test, y_pred_delta, "Delta-LSTM")


Delta-LSTM
MSE : 0.10068325698375702
RMSE: 0.31730625109467514
MAE : 0.22741062939167023
R2  : -5.090564250946045


(0.10068325698375702,
 np.float64(0.31730625109467514),
 0.22741062939167023,
 -5.090564250946045)